# Web scraping y API {#sec-webscraping-and-apis}

## Introducción

Este capítulo te mostrará cómo trabajar con datos en línea, ya sea obtenidos de páginas web mediante web scraping o de forma más directa a través de internet mediante una API. Un principio importante es usar siempre una API si hay una disponible, ya que está diseñada para pasar la información directamente a tu sesión de Python y te ahorrará mucho esfuerzo.

In [ ]:
# remove cell
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline

# Plot settings
plt.style.use("https://github.com/aeturrell/python4DS/raw/main/plot_style.txt")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

### Consideraciones éticas y legales

Como ya dijimos, deberías usar una API siempre que puedas. Pero, para los casos en que no sea posible, ¿cuáles son las reglas del juego en lo que respecta al web scraping? Primero, tenemos que hablar de si es legal y ético hacerlo. En general, la situación es complicada en ambos aspectos.

Las cuestiones legales dependen mucho de dónde vivas. Sin embargo, como principio general, si los datos son públicos, no personales, no comerciales y fácticos, probablemente no tendrás problemas. Estos tres factores son importantes porque están relacionados con los términos y condiciones del sitio, la información de identificación personal y los derechos de autor, como veremos a continuación.

Si los datos no son públicos, no son no personales o no son fácticos, o si haces scraping de los datos específicamente para ganar dinero con ellos, o si haces scraping de algo que solo está disponible comercialmente, tendrás que hablar con un abogado. En cualquier caso, debes ser respetuoso con los recursos del servidor que aloja las páginas de las que haces scraping. Sobre todo, esto significa que, si haces scraping de muchas páginas, debes asegurarte de esperar un poco entre cada solicitud. [Tenacity](https://tenacity.readthedocs.io/en/latest/) es un paquete excelente para esto: tiene una función que hace pausas entre intentos de hacer algo (además de muchas otras funcionalidades útiles).

Si te fijas bien, verás que muchos sitios web incluyen en algún lugar de la página un enlace a “términos y condiciones” o “términos de servicio”, y si lees esa página con atención a menudo descubrirás que el sitio prohíbe específicamente el web scraping. Estas páginas suelen ser un acaparamiento legal en el que las empresas hacen afirmaciones muy amplias. Es de buena educación respetar estos términos de servicio cuando sea posible, pero toma cualquier afirmación con cautela.

Los tribunales de EE. UU. han determinado en general que simplemente poner los términos de servicio en el pie de página del sitio web no basta para que quedes obligado por ellos, p. ej., [HiQ Labs v. LinkedIn](https://en.wikipedia.org/wiki/HiQ_Labs_v._LinkedIn). Por lo general, para quedar obligado por los términos de servicio, debes haber realizado alguna acción explícita, como crear una cuenta o marcar una casilla. Por eso es importante si los datos son **públicos** o no; si no necesitas una cuenta para acceder a ellos, es poco probable que estés obligado por los términos de servicio. Ten en cuenta, sin embargo, que la situación es bastante distinta en Europa, donde los tribunales han determinado que los términos de servicio son exigibles incluso si no los aceptas explícitamente.

Incluso si los datos son públicos, debes tener muchísimo cuidado al hacer scraping de información de identificación personal, como nombres, direcciones de correo electrónico, números de teléfono, fechas de nacimiento, etc. Europa tiene leyes estrictas sobre la recopilación o el almacenamiento de este tipo de datos (conocidas como RGPD o GDPR), y, vivas donde vivas, probablemente te estarás metiendo en un atolladero ético. Por ejemplo, en 2016, un grupo de investigadores hizo scraping de la información de perfiles públicos (p. ej., nombres de usuario, edad, género, ubicación, etc.) de 70.000 personas en el sitio de citas OkCupid y publicaron estos datos sin ningún intento de anonimización. Aunque los investigadores consideraron que no había nada malo en ello, ya que los datos ya eran públicos, este trabajo fue ampliamente condenado por preocupaciones éticas sobre la identificabilidad de los usuarios cuya información se publicó en el dataset. Si tu trabajo implica hacer scraping de información de identificación personal, te recomendamos encarecidamente leer sobre el estudio de OkCupid3, así como sobre estudios similares con una ética de investigación cuestionable que implicaron la obtención y publicación de información de identificación personal.

Por último, también tienes que preocuparte por la ley de derechos de autor. La ley de derechos de autor es complicada. La ley de EE. UU. describe exactamente qué está protegido: “[…] obras originales de autoría fijadas en cualquier medio tangible de expresión, […]”. Luego describe categorías específicas a las que se aplica, como obras literarias, obras musicales, películas y más. Llama la atención que los datos no gozan de protección de derechos de autor. Esto significa que, mientras limites tu scraping a hechos, la protección de derechos de autor no se aplica. (Pero ten en cuenta que Europa tiene un derecho “sui generis” aparte que protege las bases de datos).

Como breve ejemplo, en EE. UU., las listas de ingredientes y las instrucciones no son protegibles por derechos de autor, así que los derechos de autor no pueden usarse para proteger una receta. Pero si esa lista de recetas va acompañada de un contenido literario original sustancial, eso sí es protegible. Por eso, cuando buscas una receta en internet, siempre hay tanto contenido antes.

Si necesitas hacer scraping de contenido original (como texto o imágenes), es posible que aun así estés protegido por la doctrina del uso legítimo (fair use). El uso legítimo no es una regla estricta, sino que sopesa varios factores. Es más probable que se aplique si recopilas los datos con fines de investigación o no comerciales y si limitas lo que extraes a solo lo que necesitas.

### Requisitos previos

Para este capítulo necesitarás instalar el paquete **pandas**. También usaremos **seaborn**, que ya deberías tener instalado. Además, necesitarás instalar los paquetes **beautifulsoup**, **pandas-datareader** y **wbgapi** en tu terminal usando `uv add beautifulsoup4`, `uv add pandas-datareader` y `uv add wbgapi`, respectivamente. También usaremos dos paquetes integrados, **textwrap** y **requests**.

Para empezar, importemos algunos de los paquetes que necesitamos (siempre es una buena práctica importar los paquetes que necesitas al principio de un script o notebook).

In [ ]:
import textwrap

import pandas as pd
import requests
from bs4 import BeautifulSoup
from lets_plot import *

LetsPlot.setup_html()

## Extraer datos de archivos en internet con **pandas**

Es fácil leer datos de internet una vez que tienes la URL y el tipo de archivo. Aquí, por ejemplo, hay un ejemplo que lee el dataset 'storms', que está almacenado como archivo CSV en una URL (solo tomaremos las primeras 10 filas):

In [ ]:
pd.read_csv(
    "https://vincentarelbundock.github.io/Rdatasets/csv/dplyr/storms.csv", nrows=10
)

## Obtener datos mediante API

Usar una API (interfaz de programación de aplicaciones) es otra forma de obtener información de internet. Las API son simplemente una manera de que una herramienta, digamos Python, hable con otra herramienta, digamos un servidor, e intercambien información de forma útil. El caso de uso clásico sería enviar, mediante una API, una solicitud de datos que cumplan una determinada consulta y recibir a cambio una descarga de esos datos. (Siempre deberías usar preferentemente una API antes que hacer web scraping de un sitio).

Como están diseñadas para funcionar con cualquier herramienta, en realidad no necesitas un lenguaje de programación para interactuar con una API; simplemente es *mucho* más fácil si lo usas.

::: {.callout-note}
An API key is needed in order to access some APIs. Sometimes all you need to do is register with site, in other cases you may have to pay for access.
:::

Para comprobarlo, usemos directamente una API para obtener algunos datos de series temporales. Haremos la llamada a internet con el paquete **requests**.

Una API tiene un 'endpoint', la URL base, y luego una URL que codifica la pregunta. Veamos un ejemplo con la API de la ONS, cuyo endpoint es "https://api.beta.ons.gov.uk/v1/". El resto de la API tiene la forma 'data?uri=' seguida del ID largo tanto de la serie temporal (jp9z) como del dataset (LMS), que corresponde a las vacantes en el sector servicios del Reino Unido.

Los datos que devuelven las API suelen estar en formato JSON, que se parece mucho a un diccionario anidado de Python y a cuyas entradas se puede acceder de la misma manera; eso es lo que ocurre al obtener el título de la serie en el ejemplo siguiente. JSON no es bueno para el análisis, así que usaremos **pandas** para darles forma a los datos.

In [ ]:
url = "https://api.beta.ons.gov.uk/v1/data?uri=/employmentandlabourmarket/peopleinwork/employmentandemployeetypes/timeseries/jp9z/lms/previous/v108"

# Get the data from the ONS API:
json_data = requests.get(url).json()

# Prep the data for a quick plot
title = json_data["description"]["title"]
df = (
    pd.DataFrame(pd.json_normalize(json_data["months"]))
    .assign(
        date=lambda x: pd.to_datetime(x["date"]),
        value=lambda x: pd.to_numeric(x["value"]),
    )
    .set_index("date")
)

df["value"].plot(title=title, ylim=(0, df["value"].max() * 1.2), lw=3.0);

Hemos hablado de *leer* API. ¡También puedes crear la tuya propia para servir datos, modelos o lo que quieras! Es un tema avanzado y no lo cubriremos; pero si lo necesitas, la forma más sencilla es usar [Fast API](https://fastapi.tiangolo.com/). Puedes encontrar algunos tutoriales breves en video sobre Fast API [aquí](https://calmcode.io/fastapi/hello-world.html).

### Pandas Datareader: una forma más fácil de interactuar con (algunas) API

Aunque no hizo falta mucho código para obtener los datos de la ONS, sería aún mejor si fuera una sola línea, ¿verdad? Por suerte, existen algunos paquetes que lo facilitan, aunque depende de la API (y las API aparecen y desaparecen con el tiempo).

Con diferencia, la librería más completa para acceder a otras API es [**pandas-datareader**](https://pandas-datareader.readthedocs.io/en/latest/), que ofrece un acceso cómodo a:

- FRED
- Quandl
- World Bank
- OECD
- Eurostat

y más.

Veamos un ejemplo con FRED (la librería de datos económicos del Banco de la Reserva Federal de St. Louis). Esta vez, veamos la tasa de desempleo del Reino Unido:

In [ ]:
import pandas_datareader.data as web

df_u = web.DataReader("LRHUTTTTGBM156S", "fred")

df_u.plot(
    title="Desempleo en el Reino Unido (porcentaje)", legend=False, ylim=(2, 6), lw=3.0
);

Y, como también es muy útil, veamos cómo usar el paquete [**wbgapi**](https://pypi.org/project/wbgapi/) para acceder a datos del Banco Mundial. (**pandas-datareader** también solía ofrecer un lector para el Banco Mundial, pero no se ha mantenido activamente, así que preferimos **wbgapi** para trabajos nuevos).

In [ ]:
# World Bank CO2 equivalent emissions (metric tons per capita)
# https://data.worldbank.org/indicator/EN.GHG.ALL.PC.CE.AR5
# country and region codes at http://api.worldbank.org/v2/country
import wbgapi as wb

indicator_code = "EN.GHG.ALL.PC.CE.AR5"
df = (
    wb.data.DataFrame(
        indicator_code,
        ["USA", "CHN", "IND", "EAS", "ECS"],  # country and region codes
        time=range(2019, 2020),
        labels=True,
    )
    .rename(columns={"Country": "country", "YR2019": indicator_code})
    .reset_index(drop=True)
)
df["country"] = df["country"].apply(lambda x: textwrap.fill(x, 10))  # wrap long names
df = df.sort_values(indicator_code)  # re-order
df.head()

In [ ]:
(
    ggplot(df, aes(x="country", y=indicator_code))
    + geom_bar(aes(fill="country"), color="black", alpha=0.8, stat="identity")
    + scale_fill_discrete()
    + theme_minimal()
    + theme(legend_position="none")
    + ggsize(600, 400)
    + labs(
        subtitle="Gases de efecto invernadero (toneladas métricas de CO2 equivalente per cápita, 2019)",
        title="EE. UU. lidera el mundo en emisiones per cápita",
        y="",
    )
)

### La API de la OCDE

A veces conviene usar las API directamente y, por ejemplo, la API de la OCDE trae MUCHA complejidad que el acceso directo puede aprovechar. La API de la OCDE ofrece los datos en formato JSON y XML, y usaremos [**pandasdmx**](https://pandasdmx.readthedocs.io/) (también conocido como el paquete Statistical Data and Metadata eXchange (SDMX) para el ecosistema de datos de Python) para descargar los datos en formato XML y convertirlos en un dataframe normal de **pandas**.

La clave para usar la API de la OCDE es conocer sus numerosos códigos: para países, periodos, recursos y series. Puedes encontrar una orientación general sobre los códigos que usa la API [aquí](https://data.oecd.org/api/sdmx-ml-documentation/), pero dar exactamente con lo que necesitas puede ser algo complicado. Dos consejos:
1. Si sabes que lo que buscas está en un dataset con un nombre concreto, p. ej., "QNA" (Quarterly National Accounts), pon `https://stats.oecd.org/restsdmx/sdmx.ashx/GetDataStructure/QNA/all?format=SDMX-ML` en tu navegador y revisa el archivo XML; ahí puedes identificar los subcódigos y los países disponibles.
2. Navega por https://stats.oecd.org/ y usa Customise; luego marca todas las casillas "Use Codes" para ver los nombres de código de lo que estés consultando.

Veamos un ejemplo en acción. Queremos ver los datos de productividad (PIB por hora) de varios países desde 2010. Usaremos el recurso de productividad (código "PDB_LV") y queremos la medida a precios corrientes en USD (código "CPC") del PIB por trabajador ocupado (código "T_GDPEMP) desde 2010 en adelante (código "startTime=2010"). Obtendremos esto para algunos países desarrollados donde las mediciones de productividad pueden ser algo más comparables. Los comentarios de abajo explican qué ocurre en cada paso.

```python
import pandasdmx as pdmx
# Tell pdmx we want OECD data
oecd = pdmx.Request("OECD")
# Set out everything about the request in the format specified by the OECD API
data = oecd.data(
    resource_id="PDB_LV",
    key="GBR+FRA+CAN+ITA+DEU+JPN+USA.T_GDPEMP.CPC/all?startTime=2010",
).to_pandas()

df = pd.DataFrame(data).reset_index()
df.head()
```

|   | LOCATION |  SUBJECT | MEASURE | TIME_PERIOD |        value |
|--:|---------:|---------:|--------:|------------:|-------------:|
| 0 |      CAN | T_GDPEMP |     CPC |        2010 | 78848.604088 |
| 1 |      CAN | T_GDPEMP |     CPC |        2011 | 81422.364748 |
| 2 |      CAN | T_GDPEMP |     CPC |        2012 | 82663.028058 |
| 3 |      CAN | T_GDPEMP |     CPC |        2013 | 86368.582158 |
| 4 |      CAN | T_GDPEMP |     CPC |        2014 | 89617.632446 |

¡Genial, funcionó! Tenemos los datos en un formato tidy muy cómodo.

### Otras API útiles

- Hay una lista de API que se actualiza con regularidad en este [repositorio de API públicas en github](https://github.com/public-apis/public-apis). No tiene (todavía) una sección de economía, pero tiene MUCHAS otras API.
- La Berkeley Library mantiene una [lista de API de economía](https://guides.lib.berkeley.edu/c.php?g=4395&p=7995952) que vale mucho la pena revisar.
- [NASDAQ Data Link](https://docs.data.nasdaq.com/), que tiene una gran cantidad de [datos financieros](https://docs.data.nasdaq.com/docs/data-organization).
- [DBnomics](https://db.nomics.world/): datos económicos públicos proporcionados por instituciones estadísticas nacionales e internacionales, pero también por investigadores y empresas privadas.

## Web scraping

El web scraping es una forma de obtener de internet información pensada para mostrarse en un navegador. Pero solo debería usarse como último recurso, y únicamente cuando lo permitan los términos y condiciones del sitio web.

Si vas a obtener datos de internet, es mucho mejor usar una API siempre que puedas: obtener información de forma estructurada es *exactamente* la razón de ser de las API. Además, las API deberían ser más estables que los sitios web, que pueden cambiar con frecuencia. Normalmente, si una organización está de acuerdo en que obtengas sus datos, habrá creado una API expresamente para ello. Es bastante raro que exista un sitio web importante que *sí* permita el web scraping pero no tenga una API; en esos sitios, si no tienen API, lo más probable es que el scraping vaya en contra de sus términos y condiciones. Esos términos y condiciones pueden ser exigibles legalmente (las normas varían según el país, y de verdad necesitas asesoría legal si no está claro si puedes hacer scraping o no).

Hay otras razones por las que el web scraping no es tan buena idea; por ejemplo, si necesitas datos históricos, puede que estén disponibles a través de una API pero no se muestren en la página web. (O puede que no estén disponibles en absoluto, en cuyo caso lo mejor es contactar con la organización o consultar la WaybackMachine por si tomó capturas).

Así que este libro es bastante crítico con el web scraping, ya que casi siempre hay una solución mejor. Sin embargo, hay ocasiones en que resulta útil.

Si te encuentras en una situación en la que necesitas hacer scraping, asegúrate muy bien de que está permitido legalmente y también de que no infringes las reglas del `robots.txt` del sitio web: es un archivo especial presente en casi todos los sitios web que establece qué se puede rastrear legítimamente (siempre que sea legal) y en qué no deberían meterse los robots.

En Python tienes mucho donde elegir en cuanto a web scraping. Hay cinco librerías muy sólidas que cubren una amplia gama de estilos y necesidades: **requests**, **lxml**, **beautifulsoup**, **selenium** y *scrapy**.

Para un web scraping rápido y sencillo, mi combinación habitual sería **requests**, que se limita prácticamente a obtener el HTML de una página web, y **beautifulsoup**, que te ayuda a navegar por la estructura de la página y extraer lo que realmente te interesa. Para páginas web dinámicas que usan javascript en lugar de solo HTML, necesitarás **selenium**. Para escalar y acceder a miles de páginas web de forma eficiente, puedes probar **scrapy**, que puede funcionar junto con las otras herramientas y gestionar múltiples sesiones, además de todo tipo de extras... de hecho, es un "framework de web scraping".

Siempre ayuda ver el código en la práctica, así que eso haremos ahora, pero ten en cuenta que omitiremos muchos detalles importantes, como los user agents, ser "educado" con tus peticiones de scraping o ser eficiente con la caché y el rastreo.

A falta de un ejemplo mejor, hagamos scraping de la página de investigación de [http://aeturrell.com/](http://aeturrell.com/)

In [ ]:
url = "http://aeturrell.com/research"
page = requests.get(url)
page.text[:300]

Bien, ¿qué acaba de pasar? Le pedimos a requests que obtuviera el HTML de la página web y luego imprimimos los primeros 300 caracteres del texto que encontró.

Ahora vamos a convertir esto en algo que los humanos puedan leer (o leer con más facilidad) usando beautifulsoup:

In [ ]:
soup = BeautifulSoup(page.text, "html.parser")
print(soup.prettify()[60000:60500])

Ahora vemos más estructura de la página e incluso algunas *etiquetas HTML*, como 'title' y 'link'. Llegamos a la parte de extracción de datos: supongamos que queremos extraer cada párrafo de texto; podemos usar beautifulsoup para recorrer la estructura HTML y extraer solo las partes con la etiqueta de párrafo ('p').


In [ ]:
# Get all paragraphs
all_paras = soup.find_all("p")
# Just show one of the paras
all_paras[1]

Aunque este párrafo no está tan mal, puedes hacerlo más legible eliminando por completo las etiquetas HTML con el método `.text`:

In [ ]:
all_paras[1].text

Ahora supongamos que no nos interesa la mayor parte de la página y *solo* queremos obtener los nombres de los proyectos. Para ello necesitamos identificar el tipo de etiqueta del elemento que nos interesa, en este caso 'div', y su clase, en este caso "project-name". Lo hacemos así (y de paso mostramos el texto limpio):


In [ ]:
projects = soup.find_all("div", class_="project-content listing-pub-info")
projects = [x.text.strip() for x in projects]
projects

¡Hurra! Conseguimos la información que queríamos: solo necesitábamos conocer las etiquetas correctas. Un buen consejo para encontrar las etiquetas de la información que buscas es abrir la página en tu navegador (p. ej., Google Chrome), hacer clic derecho sobre la parte que te interesa y pulsar 'Inspeccionar'. Esto te mostrará el elemento HTML de la parte de la página en la que hiciste clic.

Eso es casi todo en esta brevísima introducción al web scraping. Solo veremos una cosa más: cómo iterar sobre varias páginas.

Imagina que tenemos una página web raíz como "www.codingforeconomists.com" con subpáginas como "www.codingforeconomists.com/page=1", "www.codingforeconomists.com/page=2", etc. Basta con generar iterativamente los strings HTML para pasarlos a una función que haga scraping de cada una y devuelva los datos relevantes; por ejemplo, para las primeras 50 páginas y con una función llamada `scraper()`, podrías ejecutar

```
start, stop = 0, 50
root_url = "www.codingforeconomists.com/page="
info_on_pages = [scraper(root_url + str(i)) for i in range(start, stop)]
```

Eso es todo lo que veremos aquí, pero recuerda que apenas hemos *rascado* la superficie de este tema amplio y complejo. Si quieres leer sobre una aplicación, es difícil no recomendar el artículo sobre web scraping que sin duda más ha cambiado el mundo, y que muy probablemente ha influido en tu propia vida de muchas maneras: ["The PageRank Citation Ranking: Bringing Order to the Web"](http://ilpubs.stanford.edu:8090/422/) de Page, Brin, Motwani y Winograd. Para un ejemplo más detallado de web scraping, consulta el [tutorial](https://realpython.com/python-web-scraping-practical-introduction/) de realpython.

### Web scraping de tablas

A menudo no quieres hacer scraping de una página web completa, sino que solo quieres los datos de una *tabla* dentro de la página. Por suerte, hay una forma sencilla de hacer scraping de tablas individuales con el paquete **pandas**.

Leeremos los datos de una tabla de 'https://webscraper.io/test-sites/tables' usando **pandas**. La función que usaremos es `read_html()`, que devuelve una lista de dataframes con todas las tablas que encuentra cuando le pasas una URL. Si quieres filtrar la lista de tablas, usa el argumento `match=` con un texto que solo aparezca en la(s) tabla(s) que te interesan.

El ejemplo siguiente muestra cómo funciona esto; si miramos el sitio web, vemos que la tabla que nos interesa tiene una columna 'First Name'. Por lo tanto, ejecutamos:

In [ ]:
df_list = pd.read_html("https://webscraper.io/test-sites/tables", match="First Name")
# Retrieve first entry from list of data frames
df = df_list[0]
df.head()

Esto nos deja la tabla cargada de forma ordenada en un dataframe de **pandas**, lista para seguir usándola.

Si obtienes un error '403', significa que el sitio web ha bloqueado a **pandas** porque detecta que estás haciendo web scraping. Esto ocurre porque algunas personas hacen web scraping de forma irresponsable, o porque los sitios web ofrecen otras formas preferidas de obtener los datos, por ejemplo, descargándolos completos (piensa en Wikipedia) o mediante una API. (Aun así, si de verdad lo necesitas, [a menudo puedes sortear el error 403](https://stackoverflow.com/questions/43590153/http-error-403-forbidden-when-reading-html).)